# Interactive Delay Map — Infrabel Stations

This notebook builds an **interactive, time-animated map** of mean train delays across Belgian railway stations, based on the pre-aggregated GeoJSON produced by the Data team:

- **Source file:** `../data/processed/delays_by_station_hour.geojson`
- **Granularity:** one point per `(station, hour)` pair
- **Goal:** visualize how delays evolve throughout the day (0h → 23h) using a slider animation.

Each frame of the animation corresponds to one hour. The color of every station encodes the mean delay (green = on-time, red = late).

## 1. Imports

We rely on `geopandas` to read the GeoJSON and `plotly.express` to build the interactive map. No Mapbox token is required since we use a free tile provider (`carto-positron`).

In [1]:
import geopandas as gpd
import pandas as pd
import plotly.express as px

# Renderer config for VS Code / JupyterLab / classic Jupyter.
# 'plotly_mimetype' lets VS Code render the figure natively via its built-in
# Plotly support; 'notebook_connected' is the fallback that loads plotly.js
# from a CDN so the output is also viewable in plain Jupyter and on GitHub.
import plotly.io as pio
pio.renderers.default = "plotly_mimetype+notebook_connected"

## 2. Load the GeoJSON

`geopandas.read_file` parses the GeoJSON directly into a `GeoDataFrame`, where the `geometry` column holds Shapely `Point` objects (longitude, latitude).

In [2]:
GEOJSON_PATH = "../data/processed/delays_by_station_hour.geojson"

gdf = gpd.read_file(GEOJSON_PATH)
print(f"Loaded {len(gdf)} rows — {gdf['station_label'].nunique()} unique stations")
gdf.head()

Loaded 8863 rows — 450 unique stations


,hour,mean_delay_min,n_observations,ptcarid,symbolicname,station_label,geometry
0,0,0.16,21,6,FLS,Alost,POINT (4.03868 50.94312)
1,5,1.64,77,6,FLS,Alost,POINT (4.03868 50.94312)
2,6,0.90,187,6,FLS,Alost,POINT (4.03868 50.94312)
3,7,0.75,240,6,FLS,Alost,POINT (4.03868 50.94312)
4,8,1.08,227,6,FLS,Alost,POINT (4.03868 50.94312)


## 3. Quick sanity check

Verify the columns expected by the spec are present and that hours cover the full 0–23 range.

In [3]:
print("Columns:", list(gdf.columns))
print("Hour range:", gdf["hour"].min(), "→", gdf["hour"].max())
print("Mean delay stats:")
gdf["mean_delay_min"].describe()

Columns: ['hour', 'mean_delay_min', 'n_observations', 'ptcarid', 'symbolicname', 'station_label', 'geometry']
Hour range: 0 → 23
Mean delay stats:


count    8863.000000
mean        1.712390
std         1.933916
min        -5.340000
25%         0.970000
50%         1.490000
75%         2.150000
max        88.450000
Name: mean_delay_min, dtype: float64

## 4. Prepare the dataframe for Plotly

`scatter_mapbox` expects plain `lat` / `lon` columns rather than a Shapely geometry, so we extract them from the `geometry` column. We also **sort the data by `hour`** — this is critical so that Plotly's animation slider iterates from 0h to 23h in the correct order.

In [4]:
df = pd.DataFrame({
    "station_label": gdf["station_label"],
    "hour": gdf["hour"].astype(int),
    "mean_delay_min": gdf["mean_delay_min"].astype(float),
    "n_observations": gdf["n_observations"].astype(int),
    "lon": gdf.geometry.x,
    "lat": gdf.geometry.y,
})

# Sort by hour so the animation frames are ordered 0 → 23
df = df.sort_values("hour").reset_index(drop=True)
df.head()

,station_label,hour,mean_delay_min,n_observations,lon,lat
0,Alost,0,0.16,21,4.038684,50.943123
1,Fraipont,0,1.27,16,5.723432,50.565123
2,Melsele,0,2.52,29,4.288506,51.210827
3,Merchtem,0,2.77,5,4.223681,50.953913
4,Waremme,0,0.46,30,5.248963,50.694691


## 5. Build the animated map

Configuration:
- **Color:** continuous scale from green (low delay) to red (high delay).
- **Hover tooltip:** station name, hour, exact delay, number of observations.
- **Animation:** one frame per hour via `animation_frame="hour"`.
- **Basemap:** `carto-positron` (no Mapbox API key needed).

We fix the color range using the global min/max of `mean_delay_min` so the color scale stays consistent across all animation frames — otherwise each frame would re-normalize and visual comparisons between hours would be misleading.

In [5]:
# Center the map roughly on Belgium
center_lat = df["lat"].mean()
center_lon = df["lon"].mean()

# Fixed color range so the legend is comparable across all frames.
# We clip at the 98th percentile to keep the scale readable despite outliers.
vmin = float(df["mean_delay_min"].min())
vmax = float(df["mean_delay_min"].quantile(0.98))

# Plotly 6.x: use the new MapLibre-based scatter_map (scatter_mapbox is deprecated).
# No API key is required — 'carto-positron' is a free tile style.
fig = px.scatter_map(
    df,
    lat="lat",
    lon="lon",
    color="mean_delay_min",
    size="n_observations",
    size_max=18,
    hover_name="station_label",
    hover_data={
        "hour": True,
        "mean_delay_min": ":.2f",
        "n_observations": True,
        "lat": False,
        "lon": False,
    },
    animation_frame="hour",
    color_continuous_scale=[(0.0, "green"), (0.5, "yellow"), (1.0, "red")],
    range_color=(vmin, vmax),
    zoom=7,
    center={"lat": center_lat, "lon": center_lon},
    map_style="carto-positron",
    title="Mean train delay by station and hour of day (Infrabel)",
)

# Force an explicit canvas size. VS Code's notebook output area constrains
# the rendered width otherwise, which squashes the map into a narrow column.
fig.update_layout(
    width=1400,
    height=850,
    autosize=False,
    margin={"r": 0, "t": 50, "l": 0, "b": 0},
    coloraxis_colorbar=dict(title="Mean delay (min)"),
)

# Slow the animation down a bit so the slider is comfortable to watch
fig.layout.updatemenus[0].buttons[0].args[1]["frame"]["duration"] = 800
fig.layout.updatemenus[0].buttons[0].args[1]["transition"]["duration"] = 300

fig.show(config={"responsive": True})

## 6. (Optional) Export to a standalone HTML file

Useful for sharing the map without requiring a Jupyter environment.

In [6]:
OUTPUT_HTML = "outputs/delay_map_by_hour.html"
fig.write_html(OUTPUT_HTML, include_plotlyjs="cdn", full_html=True)
print(f"Map exported to: {OUTPUT_HTML}")

Map exported to: outputs/delay_map_by_hour.html
